# Data Preprocessing 

In [1]:
import pandas as pd

### Loading all the data from source

In [2]:
df_product = pd.read_csv('../data/archive/product_info.csv') 
df_r1 = pd.read_csv('../data/archive/reviews_0-250.csv')
df_r2 = pd.read_csv('../data/archive/reviews_250-500.csv')
df_r3 = pd.read_csv('../data/archive/reviews_500-750.csv')
df_r4 = pd.read_csv('../data/archive/reviews_750-1250.csv')
df_r5 = pd.read_csv('../data/archive/reviews_1250-end.csv')
df_reviews = pd.concat([df_r1,df_r2,df_r3,df_r4,df_r5],axis=0)

C:\Users\aditi\AppData\Local\Temp\ipykernel_19612\1331934371.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_r1 = pd.read_csv('../data/archive/reviews_0-250.csv')
C:\Users\aditi\AppData\Local\Temp\ipykernel_19612\1331934371.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_r4 = pd.read_csv('../data/archive/reviews_750-1250.csv')
C:\Users\aditi\AppData\Local\Temp\ipykernel_19612\1331934371.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_r5 = pd.read_csv('../data/archive/reviews_1250-end.csv')


In [3]:
print('Shape of product df ',df_product.shape)
print('Shape of reviews df ',df_reviews.shape)

Shape of product df  (8494, 27)
Shape of reviews df  (1094411, 19)


## Cleaning product dataset

In [4]:
df_product.isna().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                 278
reviews                278
size                  1631
variation_type        1444
variation_value       1598
variation_desc        7244
ingredients            945
price_usd                0
value_price_usd       8043
sale_price_usd        8224
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights            2207
primary_category         0
secondary_category       8
tertiary_category      990
child_count              0
child_max_price       5740
child_min_price       5740
dtype: int64

### Drop columns with high null values

In [6]:
null_ratio = df_product.isnull().mean()
cols_to_drop = null_ratio[null_ratio > 0.7].index
df_product = df_product.drop(columns=cols_to_drop)

### Drop rows with NA value from reviews and rating columns

In [7]:
df_product = df_product.dropna(subset=['rating','reviews'])

In [8]:
df_product.shape

(8216, 24)

## Cleaning reviews dataset

In [9]:
df_reviews.isna().sum()

Unnamed: 0                       0
author_id                        0
rating                           0
is_recommended              167988
helpfulness                 561592
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                   1444
review_title                310654
skin_tone                   170539
eye_color                   209628
skin_type                   111557
hair_color                  226768
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
dtype: int64

### drop rows with high na

In [10]:
df_reviews = df_reviews.dropna(subset='is_recommended')

### filling helpfulness with 0 as their is no positive feedback count

In [11]:
df_reviews['helpfulness'] = df_reviews['helpfulness'].fillna(0)

## Merge reviews and product dataset on product id

In [12]:
df = df_reviews.merge(df_product,on='product_id',how='left')

In [13]:
df.shape

(926423, 42)

In [14]:
# dropping repeated columns
df = df.drop(columns=['rating_y','product_name_y','brand_name_y','price_usd_y'])

In [15]:
# renaming columns
df.columns = [col.replace('_x', '') if col.endswith('_x') else col for col in df.columns]


In [16]:
df.shape

(926423, 38)

In [17]:
common_ids = set(df_reviews['product_id']) & set(df_product['product_id'])
len(common_ids)


2351

## Saving merged dataset

In [90]:
df.to_csv('../data/processed/processed.csv', index=False)


In [19]:
df['author_id'] = df['author_id'].astype(str)

df.to_parquet('../data/processed/processed_parq.parquet', index=False)
